# 第三版完整模型时延基线（诊断）
比较同一板卡、同一原始完整Fbank下的完整ORT CPU（优化关闭/开启）与第三版55层FPGA模型。计算范围是Fbank输入→完整模型→L2→注册余弦评分，不含音频读入、Fbank生成、采集等待和首次加载。

已有55层硬件一致性通过、小子集EER/minDCF无变化，但严格QDQ分数漂移仍未通过。本次时延可以用于工程基线，不能称同精度加速。逐层参考计算和调试寄存器读回只在计时外做；计时包含实际模型内CPU计算、地址切换、DDR复制/同步和硬件调用。

20次采样分成两个块，块间反转CPU/FPGA顺序；每块每条音频先预热3次。一次只保留一个完整运行时，降低并存模型的内存影响。可能需要较长时间，实时输出进度。

In [ ]:
from pathlib import Path
import os, platform, subprocess, sys, time, uuid
sys.dont_write_bytecode=True
WORK=Path.cwd().resolve()
assert WORK.name=='benchmark_v3' and (WORK/'benchmark.py').is_file(), '请打开evaluation/benchmark_v3里的Notebook'
assert platform.system()=='Linux' and platform.machine().lower() in ('armv7l','armv8l')
RUN=WORK/'results'/('run_'+uuid.uuid4().hex)
RUN.mkdir(parents=True)
print('本次结果目录:',RUN)


In [ ]:
command=[sys.executable,'-u','-B',str(WORK/'benchmark.py'),'--output',str(RUN),'--repeats','20','--warmup','3']
logfile=RUN/'execution.log'
with logfile.open('w',encoding='utf-8') as writer:
    proc=subprocess.Popen(command,cwd=str(WORK),text=True,stdout=writer,stderr=subprocess.STDOUT,
        env={**os.environ,'PYTHONDONTWRITEBYTECODE':'1','PYTHONUNBUFFERED':'1'})
    deadline=time.monotonic()+7200
    try:
        with logfile.open('r',encoding='utf-8',errors='replace') as reader:
            while True:
                chunk=reader.read()
                if chunk: print(chunk,end='',flush=True)
                returncode=proc.poll()
                if returncode is not None:
                    tail=reader.read()
                    if tail: print(tail,end='',flush=True)
                    break
                if time.monotonic()>deadline:
                    proc.kill();proc.wait();returncode=-1
                    print('\n总时限7200秒已到，请返回日志。若IP硬件超时或仍忙，重启板卡后再继续。')
                    break
                time.sleep(2)
    finally:
        if proc.poll() is None:
            proc.kill();proc.wait()
print('\n返回码:',returncode,'结果目录:',RUN)
assert returncode==0, '测速未完成，请返回日志'


请返回最后两条V3_LATENCY_RATIO和BENCHMARK_COMPLETED。倍率>1表示FPGA更快，<1表示更慢；分别对比完整优化关闭与优化开启CPU。结果文件包含中位数、P95、冷初始化、内存/频率只读快照和向量漂移记录。不能把这个模型时延倍率写成音频端到端加速比，也不能忽略精度差异。